#### NumPy & Feature Engineering

In [ ]:
#i. Severity Labels and details for feature engineering 

import numpy as np
import pandas as pd

#summary of vehicles
vehicles = df_clean["number_of_vehicles"].to_numpy()
print("Mean:", np.mean(vehicles))
print("maximum:", np.max(vehicles))
print("Minimum:", np.min(vehicles))

# summary of casulaties
casualties = df_clean["number_of_casualties"].to_numpy()
print("Mean of casualties", np.mean(casualties))
print("Minimum of casualties", np.min(casualties))
print("Maximum of casualties", np.max(casualties))
print("Sum of casualties", np.sum(casualties))


Mean: 1.8118492982024132
maximum: 17
Minimum: 1
Mean of casualties 1.2596207830583601
Minimum of casualties 1
Maximum of casualties 142
Sum of casualties 127883


In [159]:
#ii. Feature Engineering (collison_severity)

#(A) Original severity codes converted to descriptive labels
collision_severity_label = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}

df_clean["severity_label"] = df_clean["collision_severity"].map(collision_severity_label)

#(B)  Create a Boolean collision-level KSI flag
df_clean["collision_KSI"] = df_clean["collision_severity"].isin([1,2])
print((df_clean["collision_KSI"] == True).sum())

#C. Group fatal and serious collisions together
severity_group = {
    1 : "KSI",
    2 : "KSI",
    3 : "Slight"

}
df_clean["severity_group"] = df_clean["collision_severity"].map(severity_group)
print(df_clean[["collision_severity","severity_label","collision_KSI","severity_group"]].head(10))

                                                                

26644
   collision_severity severity_label  collision_KSI severity_group
0                   3         Slight          False         Slight
1                   3         Slight          False         Slight
2                   3         Slight          False         Slight
3                   3         Slight          False         Slight
4                   3         Slight          False         Slight
5                   3         Slight          False         Slight
6                   3         Slight          False         Slight
7                   3         Slight          False         Slight
8                   3         Slight          False         Slight
9                   2        Serious           True            KSI


In [153]:
# ii. Feature Engineering (Road safety)

df_clean["multi-vehicle"] = np.where( df_clean["number_of_vehicles"]> 1, True, False)
#print(df_clean[["number_of_vehicles","multi-vehicle"]])

df_clean["higher_speed_limit"] = np.where( df_clean["speed_limit"]>= 50, True, False)
print(df_clean[["speed_limit","higher_speed_limit","number_of_vehicles","multi-vehicle"]])

#iii. High_speed_KSI
df_clean["high_speed_KSI"] = ((df_clean["higher_speed_limit"]) & (df_clean["collision_KSI"]))
print(df_clean[["higher_speed_limit","collision_KSI","high_speed_KSI"]])
print((df_clean["high_speed_KSI"] == True).sum())

        speed_limit  higher_speed_limit  number_of_vehicles  multi-vehicle
0                30               False                   1          False
1                30               False                   2           True
2                30               False                   2           True
3                40               False                   2           True
4                70                True                   7           True
...             ...                 ...                 ...            ...
101520           20               False                   1          False
101521           40               False                   2           True
101522           50                True                   2           True
101523           20               False                   2           True
101524           20               False                   2           True

[101525 rows x 4 columns]
        higher_speed_limit  collision_KSI  high_speed_KSI
0              

#### Section D — Advanced Pandas Analysis and Merging

In [ ]:
#i Calculating speed summary
speed_summary = (df_clean.groupby("speed_limit").agg( Total_collision =("collision_severity", 'size'),
                                                    Total_collision_KSI=("collision_KSI", 'sum')))

print(speed_summary)

#ii.Speed summary percentage
speed_percentage= (speed_summary["Total_collision_KSI"]/speed_summary["Total_collision"]) * 100

print(speed_percentage.round(2))

             Total_collision  Total_collision_KSI
speed_limit                                      
20                     20045                 4481
30                     49673                12323
40                      8886                 2423
50                      4612                 1364
60                     12709                 4623
70                      5600                 1430
speed_limit
20    22.35
30    24.81
40    27.27
50    29.58
60    36.38
70    25.54
dtype: float64


In [ ]:
#ii. Road Type Analysis

#Total_collisions
total_collisionsI = df_clean.groupby("first_road_class")["collision_severity"].size()
print(total_collisionsI)

#Total_collision_KSI
Total_collision_KSI = df_clean.groupby("first_road_class")["collision_KSI"].sum()
print(Total_collision_KSI)

#K_S_I percentage
road_class_summary = (df_clean.groupby("first_road_class").agg( Total_collision =("collision_severity", 'size'),
                                                    Total_collision_KSI=("collision_KSI", 'sum')))

print(road_class_summary)

# road_class_summary percentage
road_class_summary["KSI_percentage"]= (road_class_summary["Total_collision_KSI"]/road_class_summary["Total_collision"]) * 100

road_class_summary = road_class_summary.sort_values("KSI_percentage", ascending=False)

print(road_class_summary.round(2))

first_road_class
-1        6
 1     3054
 2      198
 3    43827
 4    12975
 5     4564
 6    36901
Name: collision_severity, dtype: int64
first_road_class
-1        0
 1      708
 2       57
 3    11455
 4     3714
 5     1037
 6     9673
Name: collision_KSI, dtype: int64
                  Total_collision  Total_collision_KSI
first_road_class                                      
-1                              6                    0
 1                           3054                  708
 2                            198                   57
 3                          43827                11455
 4                          12975                 3714
 5                           4564                 1037
 6                          36901                 9673
                  Total_collision  Total_collision_KSI  KSI_percentage
first_road_class                                                      
 2                            198                   57           28.79
 4               

In [ ]:
#iii. Road-class Analysis

# Road-class with highest collision
highest_collison_class =  road_class_summary["Total_collision"].idxmax()
print(highest_collison_class)

# Road class with highest KSI percentage
highest_collison_class_percentage = road_class_summary["KSI_percentage"].idxmax()
print(highest_collison_class_percentage)

3
2


In [188]:
#iv. Merging vehicles and casualty datssets

vehicles_df = pd.read_csv("dft-road-casualty-statistics-vehicle-2025.csv", dtype={"collision_index": 'string', "collision_ref_no":"string", "generic_make_model":"string"} )
casualty_df = pd.read_csv("dft-road-casualty-statistics-casualty-2025.csv", dtype={"collision_index": 'string', "collision_ref_no":"string"})

for name, data in [("vehicles", vehicles_df),("casualties", casualty_df)]:
    print(f"\n{name},dataset")
    print("shape:", data.shape)
    print("Columns:", data.columns.tolist())
    print("Unique References: ", data["collision_ref_no"].nunique())
    


vehicles,dataset
shape: (183948, 32)
Columns: ['collision_index', 'collision_year', 'collision_ref_no', 'vehicle_reference', 'vehicle_type', 'towing_and_articulation', 'vehicle_manoeuvre_historic', 'vehicle_manoeuvre', 'vehicle_direction_from', 'vehicle_direction_to', 'vehicle_location_restricted_lane_historic', 'vehicle_location_restricted_lane', 'junction_location', 'skidding_and_overturning', 'hit_object_in_carriageway', 'vehicle_leaving_carriageway', 'hit_object_off_carriageway', 'first_point_of_impact', 'vehicle_left_hand_drive', 'journey_purpose_of_driver_historic', 'journey_purpose_of_driver', 'sex_of_driver', 'age_of_driver', 'age_band_of_driver', 'engine_capacity_cc', 'propulsion_code', 'age_of_vehicle', 'generic_make_model', 'driver_imd_decile', 'lsoa_of_driver', 'escooter_flag', 'driver_distance_banding']
Unique References:  101525

casualties,dataset
shape: (127883, 24)
Columns: ['collision_index', 'collision_year', 'collision_ref_no', 'vehicle_reference', 'casualty_refere

In [ ]:
#iv.(a) Vehicles and casualty counts
vehicles_per_collision = vehicles_df.groupby("collision_ref_no").size()
casualties_per_collision = casualty_df.groupby("collision_ref_no").size()

print("highest_no_of_vehicles_per_collision:", vehicles_per_collision.max())
print("highest_no_of_casualty_per_collosion:", casualties_per_collision.max())

print("vehicle_counts_per_collision:",vehicles_per_collision.value_counts().sort_index())
print("casualty_counts_per_collision:",casualties_per_collision.value_counts().sort_index())


highest_no_of_vehicles_per_collision: 17
highest_no_of_casualty_per_collosion: 142
vehicle_counts_per_collision: 1     30841
2     61869
3      6876
4      1331
5       405
6       109
7        58
8        18
9         7
10        5
11        2
12        3
17        1
Name: count, dtype: int64
casualty_counts_per_collision: 1      83317
2      13126
3       3301
4       1178
5        389
6        113
7         46
8         17
9          9
10         9
11         3
12         6
13         1
15         1
17         1
18         2
20         2
22         1
25         1
32         1
142        1
Name: count, dtype: int64


In [199]:
#v. Unmacthed vehicle or casualty recorder in the new dataset before merging
vehicle_unmatched = (~vehicles_df["collision_ref_no"].isin(df_clean["collision_ref_no"]))
casualty_unmatched = (~casualty_df["collision_ref_no"].isin(df_clean["collision_ref_no"]))

print(vehicle_unmatched)
print(casualty_unmatched)

0         False
1         False
2         False
3         False
4          True
          ...  
183943    False
183944    False
183945     True
183946     True
183947    False
Name: collision_ref_no, Length: 183948, dtype: bool
0          True
1          True
2          True
3          True
4          True
          ...  
127878    False
127879    False
127880     True
127881    False
127882    False
Name: collision_ref_no, Length: 127883, dtype: bool


In [202]:
print("Unmatched_vehicle:",vehicle_unmatched.sum())
print("Unmatched_casualty:",casualty_unmatched.sum())

print("Collision Years:",
      df_clean["collision_year"],
      vehicles_df["collision_year"],
      casualty_df["collision_year"],

      )

Unmatched_vehicle: 54511
Unmatched_casualty: 36106
Collision Years: 0         2025
1         2025
2         2025
3         2025
4         2025
          ... 
101520    2025
101521    2025
101522    2025
101523    2025
101524    2025
Name: collision_year, Length: 101525, dtype: int64 0         2025
1         2025
2         2025
3         2025
4         2025
          ... 
183943    2025
183944    2025
183945    2025
183946    2025
183947    2025
Name: collision_year, Length: 183948, dtype: int64 0         2025
1         2025
2         2025
3         2025
4         2025
          ... 
127878    2025
127879    2025
127880    2025
127881    2025
127882    2025
Name: collision_year, Length: 127883, dtype: int64


In [205]:
print("Unamatched vehicle references :",
    (vehicles_df.loc[vehicle_unmatched, ["collision_index","collision_ref_no"]].head()))

print("Unamatched casualty references :",
    (casualty_df.loc[casualty_unmatched, ["collision_index","collision_ref_no"]].head()))


   

Unamatched vehicle references :    collision_index collision_ref_no
4    2025031672722        031672722
15   2025010566148        010566148
16   2025010553943        010553943
17   2025010603378        010603378
20   2025061538376        061538376
Unamatched casualty references :   collision_index collision_ref_no
0   2025010552583        010552583
1   2025010554623        010554623
2   2025010556322        010556322
3   2025010558573        010558573
4   2025010560329        010560329


In [206]:
print("Collision reference dtype:",
      df_clean["collision_ref_no"].dtype)

print("Vehicle reference dtype:",
      vehicles_df["collision_ref_no"].dtype)

print("Casualty reference dtype:",
      casualty_df["collision_ref_no"].dtype)

print("\nCollision reference examples:")
print(df_clean["collision_ref_no"].head(10).tolist())

print("\nVehicle reference examples:")
print(vehicles_df["collision_ref_no"].head(10).tolist())

Collision reference dtype: str
Vehicle reference dtype: string
Casualty reference dtype: string

Collision reference examples:
['17M101925', '17L205225', '70971700', '70403274', '71000604', '41649756', '161620714', '161665297', '161671173', '161631364']

Vehicle reference examples:
['552501020', '221557045', '221551102', '161606443', '031672722', '991570807', '371545889', '461626348', '201596806', '411620329']


In [208]:
# Work on copies to preserve the original columns
collision_test = df_clean.copy()
vehicle_test = vehicles_df.copy()
casualty_test = casualty_df.copy()

for data in [collision_test, vehicle_test, casualty_test]:

    refs = data["collision_ref_no"].astype("string").str.strip()

    data["join_ref"] = refs.where(
        ~refs.str.fullmatch(r"\d+").fillna(False),
        refs.str.zfill(9)
    )

print(
    "Unique normalised collision references:",
    collision_test["join_ref"].nunique()
)

print(
    "Duplicate normalised collision references:",
    collision_test["join_ref"].duplicated().sum()
)

Unique normalised collision references: 101525
Duplicate normalised collision references: 0


In [209]:
# Re-calculating the unmatched variables :
vehicle_unmatched_fixed = (
    ~vehicle_test["join_ref"].isin(collision_test["join_ref"])
)

casualty_unmatched_fixed = (
    ~casualty_test["join_ref"].isin(collision_test["join_ref"])
)

print("Unmatched vehicles:", vehicle_unmatched_fixed.sum())
print("Unmatched casualties:", casualty_unmatched_fixed.sum())

Unmatched vehicles: 0
Unmatched casualties: 0


##### vi. First validated pandas merge
**Step 1 — Understand the expected result**
Each collision may contain multiple vehicle records. Therefore, the merged table will have repeated collision information, once for every associated vehicle.

In [ ]:

#Step 2-Check collision-year consistency(data-consistancy check)

year_check = vehicle_test.merge(
    collision_test[["join_ref", "collision_year"]],
    on="join_ref",
    how="left",
    validate="many_to_one",
    suffixes=("_vehicle", "_collision")
)

year_mismatches = (
    year_check["collision_year_vehicle"]
    != year_check["collision_year_collision"]
).sum()

print("Year mismatches:", year_mismatches)


Year mismatches: 0


In [211]:
#Step 3-Perform the merge
collision_vehicle = vehicle_test.merge(
    collision_test,
    on="join_ref",
    how="left",
    suffixes=("_vehicle", "_collision"),
    validate="many_to_one",
    indicator=True
)

print("Merged shape:", collision_vehicle.shape)

print(
    collision_vehicle["_merge"].value_counts()
)

Merged shape: (183948, 84)
_merge
both          183948
left_only          0
right_only         0
Name: count, dtype: int64


In [212]:
# Step 4- Validate the merge 
print(
    "Original vehicle rows:",
    len(vehicle_test)
)

print(
    "Merged vehicle rows:",
    len(collision_vehicle)
)

print(
    "Unmatched rows:",
    (collision_vehicle["_merge"] != "both").sum()
)

print(
    "Unique collisions represented:",
    collision_vehicle["join_ref"].nunique()
)

Original vehicle rows: 183948
Merged vehicle rows: 183948
Unmatched rows: 0
Unique collisions represented: 101525


In [218]:
#vii. Integrating All Three Datasets
# aggregating the vehicle dataset, as because a single collision reference_id can contain multiple vehicle records

vehicle_summary = (vehicle_test.groupby("join_ref").agg(vehicle_count =("join_ref", "size" )).reset_index()) 
 

print(vehicle_summary)

#aggregating the casualty dataset
casualty_summary = (casualty_test.groupby("join_ref").agg(casualty_count =("join_ref", "size" )).reset_index())

print(casualty_summary)

# Merging all three 
master_collision = collision_test.merge( vehicle_summary, on="join_ref", validate="one_to_one", how="left").merge(casualty_summary, on="join_ref", validate="one_to_one", how="left"

)
print("Master_collisoin_dataset",master_collision.shape )
print("Unique_values", master_collision["join_ref"].nunique())

         join_ref  vehicle_count
0       010551784              3
1       010551786              2
2       010551792              5
3       010551794              1
4       010551795              2
...           ...            ...
101520  991732325              2
101521  991732815              2
101522  991736589              2
101523  991742597              1
101524  991744956              2

[101525 rows x 2 columns]
         join_ref  casualty_count
0       010551784               1
1       010551786               1
2       010551792               1
3       010551794               2
4       010551795               1
...           ...             ...
101520  991732325               1
101521  991732815               1
101522  991736589               1
101523  991742597               2
101524  991744956               1

[101525 rows x 2 columns]
Master_collisoin_dataset (101525, 53)
Unique_values 101525


In [222]:
#Viii. Validate the dataset now
print("Original collision: ", len(collision_test))
print("master rows: ", len(master_collision))
print("Missing vehicle counts:", master_collision["vehicle_count"].isna().sum())
print("Missing casualty counts: ",master_collision["casualty_count"].isna().sum())
print("vehicles count agree: ",master_collision["vehicle_count"].sum() == len(vehicle_test))
print("casualty count agree: ",master_collision["casualty_count"].sum() == len(casualty_test))
print("Number of unmatched vehicles: ", master_collision["vehicle_count"] != master_collision["number_of_vehicles"])


Original collision:  101525
master rows:  101525
Missing vehicle counts: 0
Missing casualty counts:  0
vehicles count agree:  True
casualty count agree:  True
Number of unmatched vehicles:  0         False
1         False
2         False
3         False
4         False
          ...  
101520    False
101521    False
101522    False
101523    False
101524    False
Length: 101525, dtype: boolean


In [225]:
# Final check 
vehicle_disparencies = (master_collision["vehicle_count"] != master_collision["number_of_vehicles"]).sum()
casualty_disparencies = (master_collision["casualty_count"] != master_collision["number_of_casualties"]).sum()

print("Vehicle disparencies: ", vehicle_disparencies)
print("casualty disparencies: ", casualty_disparencies)


Vehicle disparencies:  0
casualty disparencies:  0
